---
title: SqDRIFT
description: SqDRIFT combines two prominent algorithms, qDRIFT and SKQD, for the problem of ground-state estimation while also reducing the circuit depth.
---

# SqDRIFT algorithm for ground state estimation
*Usage estimate: ~180 seconds on ibm_aachen (NOTE: This is an estimate only. Your runtime may vary.)*

## Learning outcomes
After completing this tutorial, you can expect to understand the following information:
- How to create smaller depth circuits as compared to trotterization
- An end-to-end workflow for ground state estimation using qDRIFT and SQD
- How to use qiskit-fermions in tandem with other qiskit-addons to implement such a workflow

## Prerequisites
It is recommended that you familiarize yourself with these topics:
- [Sample based quantum diagonalization (SQD)](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-addons-sqd)
- [Sample-based Krylov Quantum Diagonalization (SKQD)](https://quantum.cloud.ibm.com/learning/en/courses/quantum-diagonalization-algorithms/skqd)

## Background

[SqDRIFT](https://arxiv.org/abs/2508.02578) is a variant of SKQD that replaces the need to choose an ansatz from which to sample bitstrings with an ensemble of time-evolution circuits constructed directly from the target hamiltonian. This is achieved by subsampling smaller time-evolution operators from said hamiltonian based on its coefficients, which is known as the qDRIFT trotterization method.

This tutorial makes use of [qiskit-fermions](https://github.com/Qiskit/qiskit-fermions) to create the more natural fermionic circuits for the [qDRIFT](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.123.070503) algorithm, followed by the use of fermionic layout and synthesis passes before plugging the circuits into the traditional qiskit pipeline for hardware execution.

Let the hamiltonian be of the form
$$
H = \sum_{i}^{N} c_i h_i
$$

Then the qDRIFT algorithm lets us realize, for the target time $t$, some operator $V_k$, where $k$ goes from $0 \cdots K$ and signifies the $k_{th}$ SqDRIFT circuit, defined as 

$$
V_k = \prod_{j=1}^{N} e^{-i h_{k_j}t \lambda / n }
$$

where 
$$
\lambda = \sum_i |c_i|
$$

and the series $(k_1, \ldots, k_n)$ is a random sequence obtained by sampling from the distribution 
$$
P[k_i] = \frac{|c_i|}{\lambda}
$$

This tutorial shows how to generate an ensemble of such randomized circuits. After we have created these circuits, similar to how we create a Krylov subspace for different operators, we sample bitstrings from multiple such operators with different time-parameters. This ensures a higher overlap between the ground-state vectors and sampled bitstrings. 


## Requirements

Before starting this tutorial, make sure you have installed
- A Python (>=3.10) virtual environment
- pip>=25.1
- qiskit ~= 2.5
- qiskit-fermions==0.1.0 (Note that the name is plural)
- numpy
- pyscf
- qiskit-aer
- qiskit-ibm-runtime
- qiskit-addon-sqd

You can install all required packages with:
```
pip install "qiskit~=2.5" "qiskit-fermions==0.1.0" qiskit-aer qiskit-ibm-runtime qiskit-addon-sqd pyscf numpy
```

## Setup

In [18]:
# Third-party scientific computing
import numpy as np

# PySCF
from pyscf import tools, ao2mo, fci

# Qiskit core
from qiskit import transpile
from qiskit.primitives import BitArray

# Qiskit Aer
from qiskit_aer import AerSimulator

# Qiskit Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# Qiskit Fermions
from qiskit_fermions.operators.library import FCIDump
from qiskit_fermions.operators import FermionOperator
from qiskit_fermions.operators.terms.filtering import filter_diagonal_terms
from qiskit_fermions.operators.terms.grouping import group_terms_by_electronic_structure
from qiskit_fermions.operators.terms.ordering import canonical_order
from qiskit_fermions.circuit import FermionicCircuit
from qiskit_fermions.circuit.library import Evolution
from qiskit_fermions.transpiler import FermionicPassManager
from qiskit_fermions.transpiler.presets import generate_preset_jw_pass_manager
from qiskit_fermions.transpiler.passes import QDriftTrotterization
from qiskit_fermions.circuit.library import InitializeModes

# Qiskit Addon SQD
from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian, SCIResult

## Simulator example

### Step 1: Map classical inputs to a quantum problem

**Reading and Preparing the FCIDump**

For this tutorial, we will load up the electronic structure hamiltonian for Nitrogen (N2). There are other ways to create Fermionic Operators as well. Please refer to the documentation at [qiskit_fermions.operators.library](https://qiskit.github.io/qiskit-fermions/stable/0.1/pydoc/qiskit_fermions.operators.library.html#module-qiskit_fermions.operators.library)

First we use the `cisolver` provided by pyscf to get the reference energy. This is the true ground-state energy of the molecule we are working with. For this we will first declare `norb` and `nelec` which are the number of orbitals and the number of electrons respectively. Then we declare `h1e` and `h2e` which are the one- and two-electron integrals respectively. All of these will later be used for SQD as well. 

In [19]:
name = "fcidump_files/N2_sto_3g"

fcidump = tools.fcidump.read(name)

# Extract metadata from the FCIDump header
norb      = fcidump["NORB"]       # number of spatial orbitals
nelec     = fcidump["NELEC"]      # total number of electrons
e_nuc     = fcidump["ECORE"]      # nuclear repulsion / core energy
ms2       = fcidump["MS2"]        # 2S (spin)

num_elec_a = (nelec + ms2) // 2   # alpha electrons
num_elec_b = (nelec - ms2) // 2   # beta  electrons

# Reconstruct full 4-index ERIs from the FCIDump (stored in 8-fold symmetry)
h1e = fcidump["H1"]           # shape (norb, norb)
h2e = ao2mo.restore(          # shape (norb, norb, norb, norb)
    1, fcidump["H2"], norb
)

cisolver = fci.direct_spin1.FCI()
cisolver.max_cycle = 200
cisolver.conv_tol  = 1e-12

e_fci, _ = cisolver.kernel(
    h1e,
    h2e,
    norb,
    (num_elec_a, num_elec_b),
    ecore=e_nuc,          # adds nuclear repulsion to the final energy
)

reference_energy = e_fci

print(f"Reference FCI Energy  = {reference_energy:.10f} Ha")

nuclear_repulsion_energy = fcidump["ECORE"]
print(f"Nuclear Repulsion Energy = {nuclear_repulsion_energy:.10f} Ha")



Parsing fcidump_files/N2_sto_3g
Reference FCI Energy  = -107.5493009579 Ha
Nuclear Repulsion Energy = 25.9296833351 Ha


**Loading the Hamiltonian**

With the necessary data ready, we read the hamiltonian from the FCI file in a format that can be fed into qiskit-fermions

In [20]:
fcidump = FCIDump.from_file(name)
hamiltonian = FermionOperator.from_fcidump(fcidump)
num_modes = 2 * fcidump.norb

**fermionic workflows using qiskit-fermions**

We will first map the hamiltonian into a fermionic circuit model using `qiskit-fermions`, which provides transpiler passes and gates specific to fermionic circuits. These will later be used before qiskit's traditional transpiler passes for this workflow.

**Term Grouping**

To ensure the reproducibility of results, we first use `canonical_order` to sort the terms based only on their structure. The order of the operators in the `canon` list is therefore fixed. This ensures reproducibility of created operators since the `QDriftTrotterization` pass that we will use in the future samples random indices to create the qDRIFT operators.

In this step, we exploit the many symmetries that are present in the electronic structure hamiltonian by grouping related terms with identical coefficients. While doing so changes the operator coefficient distribution which the qDRIFT protocol samples from, this does not affect its convergence guarantees. Crucially, the grouping of terms related by symmetry results in a favorable cancellation of Pauli terms resulting in an overall shorter circuit depth, when time-evolving a state under their action.

qiskit-fermions provides the `group_terms_by_electronic_structure` function that does this grouping for us.

Please note that the `group_terms_by_electronic_structure` assumes [normal ordering](https://qiskit.github.io/qiskit-fermions/stable/0.1/stubs/qiskit_fermions.operators.FermionOperator.html#qiskit_fermions.operators.FermionOperator.normal_ordered) terms 

`filter_diagonal_terms()` removes every term from a normal-ordered FermionOperator in place that is a product of number operators — the constant, single n_i, and higher-order n_i n_j … — since those only contribute a global phase and can't change the sampled bitstrings.


In [21]:
# Apply automatic grouping
canon = canonical_order(hamiltonian.normal_ordered().simplify(atol=1e-16))
exit_code = group_terms_by_electronic_structure(canon, num_modes, two_body_physicist_order=False)
filter_diagonal_terms(canon)

print(len(canon.groups))

4160


Now that we have grouped the terms in the hamiltonian, we will decide on the following parameters to generate the ensemble of circuits:
- the number of circuits to generate: num_circuits
- the length of each circuit in terms of excitation groups: num_exc
- the factor for the different evolution times: times

**Creating Fermionic circuits**

We will now create Fermionic circuits for each of the time-steps. Each circuit will consist of a single evolution gate, with the evolution time we declared earlier. The evolution operator is the hamiltonian. Later we run transpiler passes on these circuits to create qDRIFT circuits. 

**Ansatz preparation**

We prepare the Hartree-Fock state using the `InitializeModes` class. For Nitrogen that is simply applying X gates to the first `num_elec_a` qubits and then to the `num_elec_b` qubits, both of which equal to 7 for Nitrogen. This state represents the 7 $\alpha$ and 7 $\beta$ electrons of Nitrogen.

In [22]:
# SqDRIFT parameters
times = [1.0,10.0]           # Total evolution times used for the subspace creation
num_exc = 10         # Number of excitation groups per circuit
num_circuits = 200   # Number of circuits to generate


init_circuits = []

hf_gate = InitializeModes.from_hartree_fock(norb, (num_elec_a, num_elec_b))

for time in times :
    evo_gate = Evolution(num_modes, canon, time)
    circ = FermionicCircuit(num_modes)
    circ.append(hf_gate, circ.modes)
    circ.append(evo_gate, circ.modes)
    init_circuits.append(circ)


### Step 2: Optimize problem for quantum hardware execution 

Now that we have our circuits, we will first use the passes available in `qiskit-fermions` to perform fermionic level optimizations, followed by transpiling our circuit for the backend of choice. Since this is a simulator experiment, we will first do this for the AerSimulator

**Weight Calculation for each group**

This step is where we perform the qDRIFT sampling of terms stochastically with probabilities proportional to their coefficients in the hamiltonian. The qDRIFT transpiler pass does this for us. This helps us create shallower circuits which can be executed on the hardware more efficiently despite limited qubit connectivity, even when the hamiltonian contains long-range couplings and higher-than-quadratic terms.
After term grouping, it samples the operators based on their weights. Where for each operator $h_i$ the weight $W_{h_i}$ is defined 

$$
W_{h_i} = |c_i| / \lambda
$$

**Fermionic and Hardware native optimizations**

The function `generate_preset_jw_pass_manager()` returns a `MultiStagePassManager` that takes a FermionicCircuit and produces an optimised final circuit that we can transpile to run on our hardware. We replace its default optimization stage with a FermionicPassManager containing our QDriftTrotterization pass:

* The `QDriftTrotterization` pass uses the weight-calculation and sampling internally to generate the circuits that we will use for sampling
* The `RelabelModes` pass is another optimization pass that can be used to permute the fermionic modes to optimize connectivity across qubits and reduce gate depth. [Read more here](https://qiskit.github.io/qiskit-fermions/stable/0.1/stubs/qiskit_fermions.transpiler.passes.RelabelModes.html#qiskit_fermions.transpiler.passes.RelabelModes)

The remaining stages of the `MultiStagePassManager` run automatically and handle the full fermion-to-qubit mapping:

* [F2QLayout](https://qiskit.github.io/qiskit-fermions/stable/0.1/stubs/qiskit_fermions.transpiler.passes.TrivialF2QLayout.html): The preset pass-manager applies TrivialF2QLayout pass, which Trivially maps $n$ fermionic bits to $n$ qubits.
* [F2QSynth](https://qiskit.github.io/qiskit-fermions/stable/0.1/stubs/qiskit_fermions.transpiler.passes.F2QSynthesis.html): A transpilation pass to map fermion-based circuit instructions to qubit-based ones.

In [23]:
qdrift = QDriftTrotterization(num_exc, rng=19)

pm = generate_preset_jw_pass_manager()
pm.optimization = FermionicPassManager([qdrift])

sqdrift_circuits = []
for circ in init_circuits:
    sqdrift_circuits += (pm.run(circ) for _ in range(num_circuits))

for circ in sqdrift_circuits:
    circ.measure_all()
    
print(len(sqdrift_circuits))

400


Now that we are done with the fermionic level optimizations, we can transpile the circuits for execution on the simulator.

In [24]:
simulator = AerSimulator()
shots = 100

transpiled_circuits = transpile(sqdrift_circuits, simulator)

### Step 3: Execute using Qiskit primitives

Now that we have our circuits, we can run them using the qiskit primitives on the AerSimulator. We will combine all the counts from different circuits. We convert them to boolean vectors before finally post-processing with SQD. 

In [25]:
print(f"Executing {len(transpiled_circuits)} circuits with {shots} shots each...")

job = simulator.run(transpiled_circuits, shots=shots)
result = job.result()

all_counts = [result.get_counts(i) for i in range(len(transpiled_circuits))]

print(len(all_counts), "length before post processing")

Executing 400 circuits with 100 shots each...
400 length before post processing


### Step 4: Post-process and return result in desired classical format

**Using bitstrings for SQD**

We can now run the diagonalization scheme on the selected bit-strings to find the lowest eigenvalue that will correspond to the ground state energy of the molecule. We create a callback function, declare initial occupancies and set the parameters before finally running the diagonalization scheme. The callback function is used to print the current iteration and the current eigenvalue estimate at each iteration.

Finally to get the ground state estimate we will add the `nuclear_repulsion_energy` to the resultant energy.

**Note**: The subspace dimensions remain the same across different iterations for the simulator experiment. Since the simulator is noise-free, no configuration recovery happens, and as a consequence the subspace dimension doesn't increase. The hardware on the other hand, gives a bigger subspace for diagonalization due to noisy samples which lead to more basis vectors when doing configuration recovery. Due to this, we will also introduce another step for pruning bitstrings in the hardware section.

In [26]:
combined_counts = {}
for counts in all_counts:
    for bitstring, count in counts.items():
        combined_counts[bitstring] = combined_counts.get(bitstring, 0) + count

bit_array = BitArray.from_counts(combined_counts)
print(bit_array.num_shots)

print(f"  Alpha electrons: {num_elec_a}")
print(f"  Beta electrons: {num_elec_b}")
print(f"  Number of orbitals: {norb}")
print(f"  Number of spin orbitals (qubits): {2*norb}")
print(f"Integral shapes: h1e={h1e.shape}, h2e={h2e.shape}")

# SQD parameters
samples_per_batch = 300
num_batches = 3
max_iterations = 5

initial_occupancies = (
    np.array([1] * num_elec_a + [0] * (norb - num_elec_a)),  # alpha
    np.array([1] * num_elec_b + [0] * (norb - num_elec_b))   # beta
)

result_history = []

def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )

# Run SQD with configuration recovery
print("\nRunning SQD with configuration recovery...")
result = diagonalize_fermionic_hamiltonian(
    h1e,
    h2e,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=(num_elec_a, num_elec_b),
    num_batches=num_batches,
    energy_tol=1e-3,
    occupancies_tol=1e-3,
    max_iterations=max_iterations,
    initial_occupancies=initial_occupancies,
    seed=42,
    callback = callback
)

computed_energy = result.energy + nuclear_repulsion_energy

print("FINAL SQD RESULTS")
print(f"Orbital occupancies (alpha): {result.orbital_occupancies[0]}")
print(f"Orbital occupancies (beta): {result.orbital_occupancies[1]}")


energy_error = abs(computed_energy - reference_energy)
print(f"Reference Energy: {reference_energy:.10f} Ha")
print(f"Computed Energy:  {computed_energy:.10f} Ha")
print(f"Error:            {energy_error:.10e} Ha")



40000
  Alpha electrons: 7
  Beta electrons: 7
  Number of orbitals: 10
  Number of spin orbitals (qubits): 20
Integral shapes: h1e=(10, 10), h2e=(10, 10, 10, 10)

Running SQD with configuration recovery...
Iteration 1
	Subsample 0
		Energy: -107.54799938371386
		Subspace dimension: 5775
	Subsample 1
		Energy: -107.54799938371386
		Subspace dimension: 5775
	Subsample 2
		Energy: -107.54799938371386
		Subspace dimension: 5775
Iteration 2
	Subsample 0
		Energy: -107.54799938371386
		Subspace dimension: 5775
	Subsample 1
		Energy: -107.54799938371386
		Subspace dimension: 5775
	Subsample 2
		Energy: -107.54799938371386
		Subspace dimension: 5775
FINAL SQD RESULTS
Orbital occupancies (alpha): [0.99999402 0.99999608 0.9967006  0.99456753 0.9760022  0.97580225
 0.99514361 0.02765328 0.02736769 0.00677274]
Orbital occupancies (beta): [0.99999398 0.9999961  0.99671043 0.99456439 0.97598903 0.97580947
 0.99514378 0.02763698 0.0273721  0.00678374]
Reference Energy: -107.5493009579 Ha
Computed En

## Hardware example

Estimated hardware runtime on IBM Aachen ~ 180 seconds. 
<br> Please note that the time required on hardware may vary

The example here is restricted to 20 qubits, as once we move beyond this size, the classical sub-routine for diagonalization becomes intractable without an HPC. 

*Note:* Due to sampling error from the noise in the hardware, the subspace created for diagonalization in the hardware run will be larger than what we get when using the simulator. While it increases the dimension of the subspace we want to diagonalize, the workflow still gives us an accurate answer due to the robustness of SQD towards noise.

**Pruning of spurious strings**

Here we can choose to perform an additional step. When we have all the bitstrings from the circuit executions, we can choose to first filter out the invalid bitstrings and then proceed with SQD or move forward without the pruning. The latter is advantageous, specifically for hardware runs as that lets us perform configuration recovery and expand the subspace even further. 
Since Nitrogen can only have 7 $\alpha$ and 7 $\beta$ electrons, any bitstrings which have more or less than 7 1's in the first and the second half of the output can be discarded. We define a function that checks if the bitstrings are valid, and if not, discards them. Once we filter out the spurious bitstrings, the rest are sent into the diagonalization scheme.

In [ ]:
name = "fcidump_files/N2_sto_3g"

fcidump = tools.fcidump.read(name)

# Extract metadata from the FCIDump header
norb      = fcidump["NORB"]       # number of spatial orbitals
nelec     = fcidump["NELEC"]      # total number of electrons
e_nuc     = fcidump["ECORE"]      # nuclear repulsion / core energy
ms2       = fcidump["MS2"]        # 2S (spin)

num_elec_a = (nelec + ms2) // 2   # alpha electrons
num_elec_b = (nelec - ms2) // 2   # beta  electrons

# Reconstruct full 4-index ERIs from the FCIDump (stored in 8-fold symmetry)
h1e = fcidump["H1"]           # shape (norb, norb)
h2e = ao2mo.restore(          # shape (norb, norb, norb, norb)
    1, fcidump["H2"], norb
)

cisolver = fci.direct_spin1.FCI()
cisolver.max_cycle = 200
cisolver.conv_tol  = 1e-12

e_fci, _ = cisolver.kernel(
    h1e,
    h2e,
    norb,
    (num_elec_a, num_elec_b),
    ecore=e_nuc,          # adds nuclear repulsion to the final energy
)

reference_energy = e_fci

print(f"Reference FCI Energy  = {reference_energy:.10f} Ha")

nuclear_repulsion_energy = fcidump["ECORE"]
print(f"Nuclear Repulsion Energy = {nuclear_repulsion_energy:.10f} Ha")

fcidump = FCIDump.from_file(name)
hamiltonian = FermionOperator.from_fcidump(fcidump)
num_modes = 2 * fcidump.norb

# Apply automatic grouping
canon = canonical_order(hamiltonian.normal_ordered().simplify(atol=1e-16))
exit_code = group_terms_by_electronic_structure(canon, num_modes, two_body_physicist_order=False)
filter_diagonal_terms(canon)

print(len(canon.groups))

# SqDRIFT parameters
times = [1.0,10.0]           # Total evolution times used for the subspace creation
num_exc = 10         # Number of excitation groups per circuit
num_circuits = 200   # Number of circuits to generate

init_circuits = []
hf_gate = InitializeModes.from_hartree_fock(norb, (num_elec_a, num_elec_b))

for time in times :
    evo_gate = Evolution(num_modes, canon, time)
    circ = FermionicCircuit(num_modes)
    circ.append(hf_gate, circ.modes)
    circ.append(evo_gate, circ.modes)
    init_circuits.append(circ)

# Calculate weights for sampling (one per group)
qdrift = QDriftTrotterization(num_exc, rng=19)

pm = generate_preset_jw_pass_manager()
pm.optimization = FermionicPassManager([qdrift])

sqdrift_circuits = []
for circ in init_circuits:
    sqdrift_circuits += (pm.run(circ) for _ in range(num_circuits))

for circ in sqdrift_circuits:
    circ.measure_all()

print(len(sqdrift_circuits))

## Uncomment this code and add your API key here

# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="YOUR_TOKEN_HERE"
# )

service = QiskitRuntimeService(channel="ibm_quantum_platform")

# Select backend (choose based on qubit requirements)
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2*norb,
)

print(f"Selected backend: {backend.name} ({backend.num_qubits} qubits)")

# Transpile for hardware
transpiled_circuits = transpile(
    sqdrift_circuits,
    backend=backend,
    optimization_level=3,
    seed_transpiler=42
)

shots = 100

sampler = Sampler(mode=backend)

sampler.options.environment.job_tags = ["TUT-SqDRIFT"]

job = sampler.run(transpiled_circuits, shots=shots)
result = job.result()

# Extract counts from SamplerV2 results
all_counts = [pub_result.data.meas.get_counts() for pub_result in result]

# Set to True to filter out bitstrings that violate electron-number conservation
PRUNE = False

def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )

if PRUNE:
    all_counts_filtered = []
    for counts in all_counts:
        filtered_count = {}
        for key in counts:
            if not is_valid_bitstring(key, norb, (num_elec_a, num_elec_b)):
                continue
            elif key not in filtered_count.keys():
                filtered_count[key] = counts[key]
            else:
                filtered_count[key] += counts[key]
        all_counts_filtered.append(filtered_count)
    all_counts = all_counts_filtered

combined_counts = {}
for counts in all_counts:
    for bitstring, count in counts.items():
        combined_counts[bitstring] = combined_counts.get(bitstring, 0) + count

bit_array = BitArray.from_counts(combined_counts)
print(bit_array.num_shots)

print("Electron configuration:")
print(f"  Total electrons: {nelec}")
print(f"  Alpha electrons: {num_elec_a}")
print(f"  Beta electrons: {num_elec_b}")
print(f"  Number of orbitals: {norb}")
print(f"  Number of spin orbitals (qubits): {2*norb}")
print(f"Integral shapes: h1e={h1e.shape}, h2e={h2e.shape}")

# SQD parameters
samples_per_batch = 300
num_batches = 3
max_iterations = 5

initial_occupancies = (
    np.array([1] * num_elec_a + [0] * (norb - num_elec_a)),  # alpha
    np.array([1] * num_elec_b + [0] * (norb - num_elec_b))   # beta
)

result_history = []

def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )

# Run SQD with configuration recovery
print("\nRunning SQD with configuration recovery...")
result = diagonalize_fermionic_hamiltonian(
    h1e,
    h2e,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=(num_elec_a, num_elec_b),
    num_batches=num_batches,
    energy_tol=1e-3,
    occupancies_tol=1e-3,
    max_iterations=max_iterations,
    initial_occupancies=initial_occupancies,
    seed=42,
    callback = callback
)

computed_energy = result.energy + nuclear_repulsion_energy

print("FINAL SQD RESULTS")
print(f"Orbital occupancies (alpha): {result.orbital_occupancies[0]}")
print(f"Orbital occupancies (beta): {result.orbital_occupancies[1]}")


energy_error = abs(computed_energy - reference_energy)
print(f"Reference Energy: {reference_energy:.10f} Ha")
print(f"Computed Energy:  {computed_energy:.10f} Ha")
print(f"Error:            {energy_error:.10e} Ha")

Parsing fcidump_files/N2_sto_3g
Reference FCI Energy  = -107.5493009579 Ha
Nuclear Repulsion Energy = 25.9296833351 Ha
4160
400


Selected backend: ibm_aachen (156 qubits)
40000
Electron configuration:
  Total electrons: 14
  Alpha electrons: 7
  Beta electrons: 7
  Number of orbitals: 10
  Number of spin orbitals (qubits): 20
Integral shapes: h1e=(10, 10), h2e=(10, 10, 10, 10)

Running SQD with configuration recovery...
Iteration 1
	Subsample 0
		Energy: -107.54366531313084
		Subspace dimension: 7896
	Subsample 1
		Energy: -107.53835584929567
		Subspace dimension: 8342
	Subsample 2
		Energy: -107.54813170270293
		Subspace dimension: 7031
Iteration 2
	Subsample 0
		Energy: -107.54914619320797
		Subspace dimension: 8742
	Subsample 1
		Energy: -107.54922291461097
		Subspace dimension: 9118
	Subsample 2
		Energy: -107.54918704316503
		Subspace dimension: 8740
Iteration 3
	Subsample 0
		Energy: -107.54926092635824
		Subspace dimension: 9600
	Subsample 1
		Energy: -107.54926960809786
		Subspace dimension: 9120
	Subsample 2
		Energy: -107.54927953582288
		Subspace dimension: 8455
FINAL SQD RESULTS
Orbital occupancies (

## Next Steps

<Admonition type="tip">

If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Sample-based quantum diagonalization of a chemistry hamiltonian](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization) - a tutorial on how to construct local unitary cluster Jastrow (LUCJ) circuit for quantum chemistry simulation.
- The [SqDRIFT](https://arxiv.org/abs/2508.02578) paper - the literature that this tutorial is based upon (Please note that some of the optimizations discussed in this paper are currently a work in progress, this tutorial is subject to change in the future based on the evolution of the used libraries).